In [1]:
from mmpose.apis import MMPoseInferencer
from datetime import datetime
import subprocess
import os
import json
import glob
import cv2

# Only include the following code if you are running the complete cloned project
while os.getcwd().split('/')[-1] != 'mmpose-synthetic-tune':
    os.chdir('..')

/opt/conda/lib/python3.11/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


In [2]:
coco_cow_category = {
    "id": 1,
    "name": "cow",
    "supercategory": "",
    "keypoints": [
        "Back1",
        "Back2",
        "Back3",
        "Back4",
        "Head",
        "Nose",
        "Neck",
        "L_Shoulder",
        "L_Elbow",
        "L_F_Paw",
        "R_Shoulder",
        "R_Elbow",
        "R_F_Paw",
        "Belly",
        "L_Hip",
        "L_Knee",
        "L_H_Paw",
        "R_Hip",
        "R_Knee",
        "R_H_Paw"
    ],
    "skeleton": [
        [
            3,
            4
        ],
        [
            16,
            17
        ],
        [
            12,
            13
        ],
        [
            8,
            9
        ],
        [
            11,
            14
        ],
        [
            18,
            1
        ],
        [
            6,
            5
        ],
        [
            18,
            19
        ],
        [
            14,
            18
        ],
        [
            14,
            15
        ],
        [
            9,
            10
        ],
        [
            8,
            14
        ],
        [
            1,
            2
        ],
        [
            19,
            20
        ],
        [
            15,
            1
        ],
        [
            15,
            16
        ],
        [
            7,
            6
        ],
        [
            4,
            7
        ],
        [
            11,
            12
        ],
        [
            2,
            3
        ],
        [
            7,
            11
        ],
        [
            7,
            8
        ]
    ]
    }

In [3]:
class MMPoseModelCoach:
    command = 'python'
    script = 'mmpose/tools/train.py'
    
    detector_model = {  #rtmdet
        "det_model": 'mmdetection/configs/rtmdet/rtmdet_l_swin_b_p6_4xb16-100e_coco.py',
        "det_weights": 'checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth'
    }

    def __init__(self, config=None, resume=True, work_dir=None, notes=''):
        self.creation_time = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')

        self.config = config
        self.resume = '--resume' if resume == True else ''
        if work_dir is not None:
            self.work_dir = work_dir
            self.config_path = glob.glob(f'{self.work_dir}/*.py')[0]
        else:
            self.config_path = f'dataset-coco/custom-configs/{self.config}'
            self.work_dir = f'models/_train-{self.creation_time}-{notes}'

        self.args = [
            self.command,
            self.script,
            self.config_path,
            '--work-dir',
            self.work_dir,
            self.resume,
        ]

    def train(self):
        subprocess.run(self.args)

    def visualize_results(self, vis_input, model_ckpt=None, radius=4, thickness=1):
        if model_ckpt is None:
            checkpoint_path = glob.glob(f'{self.work_dir}/best*.pth')[0]
        else:
            checkpoint_path = f'{self.work_dir}/{model_ckpt}'
        
        poser_model = {
            "pose2d": self.config_path,
            "pose2d_weights": checkpoint_path
        }

        inferencer = MMPoseInferencer(**poser_model, **self.detector_model, device='cuda:0')

        self.input_path = vis_input
        self.output_path = f'{self.work_dir}/vis_results'

        result_generator = inferencer(
            self.input_path,
            radius=radius,
            thickness=thickness,
            vis_out_dir=self.output_path,
            draw_heatmap=True,
            det_cat_ids=5
        )

        self.results = [res for res in result_generator]
        return self.results
        
    def results_to_coco(self):
        if not hasattr(self, 'results'):
            print('Run visualize_results first')
            return
        
        # Creating images
        all_images = []
        if os.path.isdir(self.input_path):
            file_names = sorted(glob.glob(f'{self.input_path}/*'))
        else:
            file_names = [self.input_path]
            
        id = 0
        for file_name in file_names:
            img = cv2.imread(file_name)
            h, w = img.shape[:2]
            img_coco = {
                "id": id,
                "width": w,
                "height": h,
                "file_name": file_name.split('/')[-1],
                "license": 0,
                "flickr_url": "",
                "coco_url": "",
                "date_captured": 0
            }
            all_images.append(img_coco)
            id += 1
            
        
        # Creating annotations
        all_anns = []
        kp_groups = list(map(lambda r: r['predictions'][0][0]['keypoints'], self.results))
        kp_score_groups = list(map(lambda r: r['predictions'][0][0]['keypoint_scores'], self.results))
        id = 0
        for kps, scores in zip(kp_groups, kp_score_groups):
            # base for coco keypoint annotations
            ann_coco = {
                "id": id,
                "image_id": id,
                "category_id": 1,
                "segmentation": [],
                "area": 0,
                "bbox": [],
                "iscrowd": 0,
                "attributes": {
                    "occluded": False,
                    "keyframe": False
                },
                "keypoints": [],
                "num_keypoints": 0
            }
            
            # adding keypoints in coco format
            ann_kps = []
            visible_kps = []
            for kp, score in zip(kps, scores):
                visibility = 2 if score >= .3 else 0
                ann_kps.extend([*kp, visibility])
                if visibility == 2:
                    visible_kps.extend(kp)

            ann_coco['keypoints'] = ann_kps
            
            # Calculating bounding box
            visible_kp_xs = [visible_kps[i] for i in range(0, len(visible_kps), 2)]
            visible_kp_ys = [visible_kps[i] for i in range(1, len(visible_kps), 2)]
            bbox_x = min(visible_kp_xs)
            bbox_y = min(visible_kp_ys)
            bbox_w = max(visible_kp_xs) - bbox_x
            bbox_h = max(visible_kp_ys) - bbox_y
            ann_coco['bbox'] = [ bbox_x, bbox_y, bbox_w, bbox_h ]
            
            # Calculating area
            ann_coco['area'] = bbox_w * bbox_h
            all_anns.append(ann_coco)
            id += 1
            
        coco_dataset = {
            "images": all_images,
            "annotations": all_anns,
            "categories": [coco_cow_category]
        }
        
        with open(f'{self.output_path}/results_coco_{self.creation_time}.json', 'w') as f:
            json.dump(coco_dataset, f)
        
        return coco_dataset
        
        

### Train and Visualize

In [12]:
_20kp_synthetic = MMPoseModelCoach(
    config='td-hm_hrnet-w48_8xb64-256x256-ap10k_finetune-20kp-synthetic.py',
    notes='synthetic-test'
)

_20kp_synthetic.train()

/opt/conda/lib/python3.11/site-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


08/06 20:58:51 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.11.9 (main, Apr 19 2024, 16:48:06) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 538918698
    GPU 0: NVIDIA GeForce RTX 4070
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.4, V12.4.131
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.4.0
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2023.1-Product Build 20230303 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.4.2 (Git Hash 1137e04ec0b5251ca2b4400a4fd3c667ce843d67)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 12.4
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-genco

/opt/conda/lib/python3.11/site-packages/mmengine/runner/checkpoint.py:347: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(filename, map_location=map_l

08/06 20:58:53 - mmengine - WARNING - The model and loaded state dict do not match exactly

unexpected key in source state_dict: head.0.0.0.conv1.weight, head.0.0.0.bn1.weight, head.0.0.0.bn1.bias, head.0.0.0.bn1.running_mean, head.0.0.0.bn1.running_var, head.0.0.0.conv2.weight, head.0.0.0.bn2.weight, head.0.0.0.bn2.bias, head.0.0.0.bn2.running_mean, head.0.0.0.bn2.running_var, head.0.0.0.conv3.weight, head.0.0.0.bn3.weight, head.0.0.0.bn3.bias, head.0.0.0.bn3.running_mean, head.0.0.0.bn3.running_var, head.0.0.0.downsample.0.weight, head.0.0.0.downsample.1.weight, head.0.0.0.downsample.1.bias, head.0.0.0.downsample.1.running_mean, head.0.0.0.downsample.1.running_var, head.0.1.0.conv1.weight, head.0.1.0.bn1.weight, head.0.1.0.bn1.bias, head.0.1.0.bn1.running_mean, head.0.1.0.bn1.running_var, head.0.1.0.conv2.weight, head.0.1.0.bn2.weight, head.0.1.0.bn2.bias, head.0.1.0.bn2.running_mean, head.0.1.0.bn2.running_var, head.0.1.0.conv3.weight, head.0.1.0.bn3.weight, head.0.1.0.bn3.bias, hea

In [14]:
_20kp.visualize_results(
    vis_input='/home/galiold/Desktop/istockphoto-104783196-612x612.jpg',
    thickness=2,
    radius=7
)

Loads checkpoint by local backend from path: models/_train-2024-04-24_13-48-52-20kp-train-on-ap10k-w48-adjusted-labels/best_coco_AP_epoch_110.pth
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
04/24 13:55:48 - mmengine - INFO - the output image has been saved at models/_train-2024-04-24_13-48-52-20kp-train-on-ap10k-w48-adjusted-labels/vis_results/test.jpg


### Visualize Only

In [4]:
vis_input = '/home/galiold/projects/datasets/test-footage/rendered'

In [5]:
_20kp_on_ap10k = MMPoseModelCoach(
    work_dir='models/trained-20kp-on-ap10k'
)
_20kp_on_ap10k.visualize_results(
    # model_ckpt='epoch_210.pth',
    vis_input=vis_input
)
dataset = _20kp_on_ap10k.results_to_coco()

Loads checkpoint by local backend from path: models/trained-20kp-on-ap10k/best_coco_AP_epoch_110.pth
05/01 13:09:43 - mmengine - WARNING - Failed to search registry with scope "mmpose" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmpose" is a correct scope, or whether the registry is initialized.
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
05/01 13:09:46 - mmengine - WARNING - Failed to search registry with scope "mmdet" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet" is a correct scope, or whether the registry is initialized.


/home/galiold/projects/global_venvs/dl-3.9/lib/python3.9/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3549.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


05/01 13:09:46 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img010.png
05/01 13:09:47 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img013.png
05/01 13:09:47 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img016.png
05/01 13:09:48 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img017.png
05/01 13:09:48 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img019.png
05/01 13:09:49 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img022.png
05/01 13:09:49 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img026.png
05/01 13:09:49 - mmengine - INFO - the output image has been saved at models/trained-20kp-on-ap10k/vis_results/6_img029.png
05/01 13

ValueError: min() arg is an empty sequence

In [12]:
pretrained_ap10k = MMPoseModelCoach(
    work_dir='models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029'
)
pretrained_ap10k.visualize_results(
    model_ckpt='hrnet_w48_ap10k_256x256-d95ab412_20211029.pth',
    vis_input=vis_input
)

Loads checkpoint by local backend from path: models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029/hrnet_w48_ap10k_256x256-d95ab412_20211029.pth
Loads checkpoint by local backend from path: checkpoints/rtmdet_l_swin_b_p6_4xb16-100e_coco-a1486b6f.pth
04/23 15:11:58 - mmengine - INFO - the output image has been saved at models/pretrained-hrnet_w48_ap10k_256x256-d95ab412_20211029/vis_results/test-29.png
